# Deploy Halos Outside-In Safety — SIL (Software-in-the-Loop)

One-click deployment of the **Halos Outside-In Safety** reference architecture in **SIL** mode on a GPU cloud instance (designed for NVIDIA Brev). Full product docs: [NVIDIA Halos Outside-In Safety](https://docs.nvidia.com/halos-outside-in/latest/index.html).

SIL runs the full **closed safety loop** on a single host across **two Docker Compose stacks**:

![Halos SIL warehouse architecture](../../assets/sil_warehouse_architecture.png)

**Stack 1 — VSS Warehouse 3.2** (perception backend): pulled + deployed first, with SIL-specific overrides.
**Stack 2 — Halos SIL**: `safety-core` (PSF), `comm-layer` (UDP→ROS bridge), `isaac-sim`, `forklift-controller` (ROS 2 waypoint driver + MUTE/UNMUTE actuation).

**What this notebook does**
1. Validates prerequisites (dual-GPU w/ RT cores, driver, Docker, toolkit) and applies kernel `sysctl` tuning
2. Installs/configures NGC CLI, logs in to `nvcr.io`
3. Fetches artifacts: VSS app-data, Halos **sil-data**, pre-pulls **PSF** + **Isaac Sim** images
4. Applies **VSS SIL overrides** (Kafka profile, SEI/NTP DeepStream fixes)
5. Deploys + verifies **VSS perception** (3 cams @ ~30 FPS + Kafka `mdx-events`)
6. Deploys + verifies **Halos SIL** (PSF wired, ROS isolated)
7. Runs the **Isaac Sim scenario** and confirms **sim-driven MUTE/UNMUTE**
8. Prints **VST UI** access links; provides stop / teardown

> First full run takes **~30–45 min** (image pulls + TensorRT engine build + Isaac RT shader compile). Subsequent runs are much faster (caches warm).


## 1. Configuration

Set your NGC API key and paths below. All subsequent cells use these variables.

**Required to set yourself:**
- `NGC_CLI_API_KEY` — get from https://ngc.nvidia.com (needs access to the `nvidia/halos-outside-in` team). Leave empty to auto-load from `~/.ngc/config` if already configured on the instance.

**Repo paths** (Brev clones both repos automatically — override if elsewhere):
- `VSS_REPO_PATH` — the `video-search-and-summarization` checkout (perception)
- `HALOS_REPO_PATH` — the `halos-outside-in-safety` checkout (safety)


In [ ]:
# ============================================================
# REQUIRED
# ============================================================
NGC_CLI_API_KEY = ""        # NGC key with nvidia/halos-outside-in access. Empty = load from ~/.ngc/config.
NGC_CLI_ORG     = ""        # Optional NGC org. Empty = keep existing ~/.ngc/config / infer.

# ---- Repo paths (auto-detected: tries these, then ~/halos/<name>) ----
VSS_REPO_PATH   = "~/video-search-and-summarization"
HALOS_REPO_PATH = "~/halos-outside-in-safety"

# ---- Hardware profile (must match VSS blueprint_config.yml) ----
# RTXPRO6000BW | RTXA6000ADA | RTXA6000 | H100 | L40S | L40 | L4
HARDWARE_PROFILE = "RTXA6000ADA"

# ---- GPU assignment ----
PERCEPTION_GPU    = 0     # GPU for VSS perception
ISAAC_GPU_DEVICE  = -1    # GPU for Isaac Sim (needs RT cores + >20GB). -1 = auto (1 if >1 GPU else 0)

# ---- ROS domain (unique per machine, 0-232) ----
ROS_DOMAIN_ID = 42

# ---- NGC resources ----
APP_DATA_RESOURCE = "nvidia/vss-warehouse/vss-warehouse-app-data:3.2.0"  # VSS app-data
SIL_DATA_RESOURCE = ""   # Halos sil-data. Empty = use MDX_DATA_RESOURCE pinned in sil.env

# ============================================================
# OPTIONAL
# ============================================================
DOWNLOAD_DIR          = ""   # NGC artifact dir (empty = home dir)
HOST_IP_OVERRIDE      = ""   # internal IP (empty = auto-detect)
EXTERNAL_IP_OVERRIDE  = ""   # external/browser IP (empty = auto-detect)
SIMULATION_DURATION   = 1200.0 # Isaac scenario length in SECONDS (~20 min) -- keeps the sim live through Sections 13-15.1; set 0 to use the scene default (simulation_duration: 20.0 in default_config_ros.yaml)

In [ ]:
# ---- Validate & resolve configuration ----
import os, re, subprocess

def _resolve_repo(default_path, name):
    """Return an existing repo path: the configured default, else ~/halos/<name>."""
    cand = os.path.expanduser(default_path)
    if os.path.isdir(cand):
        return cand
    alt = os.path.expanduser(os.path.join("~/halos", name))
    if os.path.isdir(alt):
        return alt
    raise AssertionError(
        f"{name} not found at {cand!r} or {alt!r}. Set the path in Section 1 "
        f"(on Brev the repo is cloned automatically)."
    )

def _env_value(env_path, key):
    """Read KEY=value (strip quotes / inline comment) from an env file."""
    if not os.path.isfile(env_path):
        return ""
    for line in open(env_path):
        s = line.strip()
        if s.startswith(key + "="):
            v = s.split("=", 1)[1]
            v = v.split(" #", 1)[0].strip().strip("'").strip('"')
            return v
    return ""

VSS_REPO_PATH   = _resolve_repo(VSS_REPO_PATH, "video-search-and-summarization")
HALOS_REPO_PATH = _resolve_repo(HALOS_REPO_PATH, "halos-outside-in-safety")

# VSS layout
VSS_DEPLOY_DIR = os.path.join(VSS_REPO_PATH, "deploy", "docker")
WH_OPS_DIR     = os.path.join(VSS_DEPLOY_DIR, "industry-profiles", "warehouse-operations")
WAREHOUSE_ENV  = os.path.join(WH_OPS_DIR, ".env")
DS_MAIN_CONFIG = os.path.join(WH_OPS_DIR, "warehouse-2d-app", "deepstream", "configs", "ds-main-config.txt")
VST_CONFIG     = os.path.join(WH_OPS_DIR, "warehouse-2d-app", "vst", "configs", "vst_config.json")
assert os.path.isfile(os.path.join(VSS_DEPLOY_DIR, "compose.yml")), f"VSS compose.yml not found under {VSS_DEPLOY_DIR}"
assert os.path.isfile(WAREHOUSE_ENV), f"VSS warehouse .env not found: {WAREHOUSE_ENV}"

# Halos layout
HALOS_DEPLOY_DIR = os.path.join(HALOS_REPO_PATH, "deployments")
SIL_ENV          = os.path.join(HALOS_DEPLOY_DIR, "profiles", "sil.env")
assert os.path.isfile(os.path.join(HALOS_DEPLOY_DIR, "compose.yaml")) or os.path.isfile(os.path.join(HALOS_DEPLOY_DIR, "compose.yml")), f"Halos compose not found under {HALOS_DEPLOY_DIR}"
assert os.path.isfile(SIL_ENV), f"Halos sil.env not found: {SIL_ENV}"

# Images + sil-data resource come from sil.env (single source of truth)
PSF_IMAGE        = _env_value(SIL_ENV, "PSF_IMAGE")
ISAAC_SIM_IMAGE  = _env_value(SIL_ENV, "ISAAC_SIM_IMAGE")
MDX_DATA_RESOURCE = SIL_DATA_RESOURCE or _env_value(SIL_ENV, "MDX_DATA_RESOURCE")
assert PSF_IMAGE and ISAAC_SIM_IMAGE, "Could not read PSF_IMAGE / ISAAC_SIM_IMAGE from sil.env"

# NGC key: from Section 1, else ~/.ngc/config
if not NGC_CLI_API_KEY:
    cfg = os.path.expanduser("~/.ngc/config")
    if os.path.isfile(cfg):
        for line in open(cfg):
            if line.strip().startswith("apikey"):
                NGC_CLI_API_KEY = line.split("=", 1)[1].strip()
                break
assert NGC_CLI_API_KEY, "NGC_CLI_API_KEY required (set in Section 1 or configure ~/.ngc/config)."
os.environ["NGC_CLI_API_KEY"] = NGC_CLI_API_KEY

# GPU count -> resolve ISAAC_GPU_DEVICE auto
_n = subprocess.run(["bash", "-c", "nvidia-smi --query-gpu=index --format=csv,noheader | wc -l"],
                    capture_output=True, text=True)
NUM_GPUS = int(_n.stdout.strip() or "1")
if ISAAC_GPU_DEVICE < 0:
    ISAAC_GPU_DEVICE = 1 if NUM_GPUS > 1 else 0

# App-data + sil-data dirs (resolved fully in Section 6)
_base = DOWNLOAD_DIR.rstrip("/") if DOWNLOAD_DIR else os.path.expanduser("~")
_app_ver = APP_DATA_RESOURCE.split(":", 1)[-1]
APP_DATA_DIR = os.path.join(_base, f"vss-warehouse-app-data_v{_app_ver}", "vss-warehouse-app-data")
MDX_DATA_DIR = os.path.join(_base, "sil-data")

print("Configuration:")
print(f"  VSS_REPO_PATH:     {VSS_REPO_PATH}")
print(f"  HALOS_REPO_PATH:   {HALOS_REPO_PATH}")
print(f"  HARDWARE_PROFILE:  {HARDWARE_PROFILE}")
print(f"  GPUs detected:     {NUM_GPUS}  (perception=GPU{PERCEPTION_GPU}, Isaac=GPU{ISAAC_GPU_DEVICE})")
print(f"  ROS_DOMAIN_ID:     {ROS_DOMAIN_ID}")
print(f"  APP_DATA_RESOURCE: {APP_DATA_RESOURCE}")
print(f"  MDX_DATA_RESOURCE: {MDX_DATA_RESOURCE}")
print(f"  PSF_IMAGE:         {PSF_IMAGE}")
print(f"  ISAAC_SIM_IMAGE:   {ISAAC_SIM_IMAGE}")
print(f"  NGC key:           {NGC_CLI_API_KEY[:4]}...{NGC_CLI_API_KEY[-4:]}")
print("\nConfiguration valid.")

### Shared helpers

Defines `run()`, `stream_compose()` (streams `docker compose` progress to a log + live status), `poll()`, and `heal_infra_perms()` (auto-fixes the data-dir permission issues that crash Kafka/Redis/Elasticsearch). Run this once.


In [ ]:
import os, re, time, shlex, subprocess

def run(cmd, cwd=None, check=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    if check and r.returncode != 0:
        raise RuntimeError(f"Command failed: {cmd}\n{r.stderr}\n{r.stdout}")
    return r.stdout.strip()

def stream_compose(cmd, cwd, log_file, extra_env=None):
    """Run a long docker compose command, stream to log_file, print periodic status."""
    env = {**os.environ, "NO_COLOR": "1", "NGC_CLI_API_KEY": NGC_CLI_API_KEY}
    if extra_env:
        env.update(extra_env)
    print(f"$ {' '.join(cmd)}\n  cwd={cwd}  log={log_file}")
    t0 = time.monotonic(); last = 0.0
    containers, errors = {}, []
    CRE = re.compile(r"^\s*Container\s+(\S+)\s+(.+?)\s*$")
    p = subprocess.Popen(cmd, cwd=cwd, env=env, stdin=subprocess.DEVNULL,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    with open(log_file, "w") as log:
        for line in p.stdout:
            log.write(line); log.flush()
            s = line.rstrip()
            if "[ERROR]" in s or "failed to start" in s:
                errors.append(s)
            m = CRE.match(s)
            if m:
                st = m.group(2)
                containers[m.group(1)] = "Exited" if st.startswith("Exited") else st
            now = time.monotonic()
            if now - last > 12:
                last = now
                tail = "; ".join(f"{n}={st}" for n, st in list(containers.items())[-5:])
                print(f"  [{now - t0:.0f}s] containers={len(containers)} errors={len(errors)} | {tail}")
        rc = p.wait()
    print(f"  finished rc={rc} in {time.monotonic() - t0:.0f}s ({len(errors)} error line(s))")
    return rc

def poll(check_fn, timeout_s, interval_s, desc):
    """Poll check_fn() until truthy or timeout. Returns True/False."""
    t0 = time.monotonic()
    while time.monotonic() - t0 < timeout_s:
        try:
            if check_fn():
                print(f"  [OK] {desc} ({time.monotonic() - t0:.0f}s)")
                return True
        except Exception as e:
            print(f"  (poll error: {e})")
        print(f"  [{time.strftime('%H:%M:%S')}] waiting: {desc} ...")
        time.sleep(interval_s)
    print(f"  [FAIL] TIMEOUT after {timeout_s}s: {desc}")
    return False

def heal_infra_perms():
    """Fix data-dir/volume permissions that make Kafka/Redis/Elasticsearch crash-loop.

    The VSS app-data bind mounts (redis) and named volumes (kafka/elastic) can be
    owned such that the container UID can't write -> 'Permission denied'. chmod 777
    the app-data dir + the offending volume sources, drop the stale redis.log, restart.
    """
    for _ in range(3):
        ps = run("docker ps --format '{{.Names}} {{.Status}}'", check=False)
        bad = [n for n in ("kafka", "redis", "elasticsearch")
               if any(l.startswith(n + " ") and "Restarting" in l for l in ps.splitlines())]
        if not bad:
            return True
        print(f"  healing infra perms for: {bad}")
        run(f"sudo chmod -R 777 {shlex.quote(APP_DATA_DIR)}", check=False)
        for c in bad:
            srcs = run("docker inspect " + c + " --format '{{range .Mounts}}{{if eq .Type \"volume\"}}{{.Source}} {{end}}{{end}}'", check=False)
            for src in srcs.split():
                run(f"sudo chmod -R 777 {shlex.quote(src)}", check=False)
        run(f"sudo rm -f {shlex.quote(os.path.join(APP_DATA_DIR, 'data_log/redis/log/redis.log'))}", check=False)
        run("docker restart " + " ".join(bad), check=False)
        time.sleep(18)
    ps = run("docker ps --format '{{.Names}} {{.Status}}'", check=False)
    return not any(("Restarting" in l) for l in ps.splitlines()
                   if l.split(" ")[0] in ("kafka", "redis", "elasticsearch"))

print("Helpers ready: run(), stream_compose(), poll(), heal_infra_perms()")

## 2. Prerequisites Check (+ apply sysctl)

Verifies and (where safe) fixes host requirements:
- **GPU**: driver ≥ 580.95.05, ≥ 2 usable GPUs ideally (perception + Isaac w/ RT cores). Single 48 GB RTX 6000 Ada also works.
- **Docker**: in tested range **[28.3.3, 29.5.0)**, non-root, Compose ≥ 2.39
- **NVIDIA Container Toolkit**: `nvidia` runtime usable
- **Kernel sysctl** (Elasticsearch/Kafka): `vm.max_map_count=262144`, `net.core.rmem_max/wmem_max=5242880` — **applied here**
- **Resources**: 16+ cores, 32 GB+ RAM, 200 GB+ free disk

> ⚠️ Docker **29.5.0+ breaks NGC image pulls**. If detected, pin Docker to 28.3.3 before deploying.


In [ ]:
import subprocess

print("=== 2.1 GPU & driver ===")
print(run("nvidia-smi --query-gpu=index,name,driver_version,memory.total --format=csv,noheader"))
print()

print("=== 2.2 Docker / Compose ===")
print(run("docker --version", check=False))
print(run("docker compose version", check=False))
print("docker non-root:", "OK" if subprocess.run(["docker","ps"],capture_output=True).returncode==0 else "NEEDS sudo (fix: sudo usermod -aG docker $USER && newgrp docker)")
_dv = run("docker version --format '{{.Server.Version}}'", check=False)
def _ver_tuple(v):
    import re as _re
    return tuple(int(x) for x in _re.findall(r"\d+", v)[:3]) if v else (0,0,0)
if _dv:
    if (28,3,3) <= _ver_tuple(_dv) < (29,5,0):
        print(f"  Docker {_dv}: in tested range [28.3.3, 29.5.0)")
    else:
        print(f"  WARNING: Docker {_dv} outside tested range [28.3.3, 29.5.0). 29.5.0+ breaks NGC pulls -- pin to 28.3.3.")
print()

print("=== 2.3 NVIDIA Container Toolkit ===")
_tk = subprocess.run(["bash","-c","docker run --rm --gpus all nvidia/cuda:12.6.3-base-ubuntu24.04 nvidia-smi >/dev/null 2>&1"]).returncode
print("  toolkit:", "OK" if _tk==0 else "NOT working -- see prerequisites.md auto-install")
print()

print("=== 2.4 Kernel sysctl (applying) ===")
print(run("sudo sysctl -w vm.max_map_count=262144 net.core.rmem_max=5242880 net.core.wmem_max=5242880"))
print()

print("=== 2.5 Firewall (ufw): allow docker bridge -> host services ===")
# VSS runs redis/kafka/etc. on the host network; some init containers (e.g.
# sdrc-wait-for-redis) sit on a docker *bridge* and reach those via HOST_IP.
# Brev VMs ship ufw active (default-deny INBOUND) which silently drops that
# bridge->host traffic, so the perception streams never start. Allow the docker
# private ranges to the host (container-internal only -- nothing public is exposed).
_ufw = run("sudo ufw status 2>/dev/null | head -1", check=False)
if "Status: active" in _ufw:
    run("sudo ufw allow from 172.16.0.0/12 comment 'docker bridge -> host (VSS)'", check=False)
    run("sudo ufw allow from 192.168.0.0/16 comment 'docker bridge -> host (VSS)'", check=False)
    print("  ufw active -> allowed 172.16.0.0/12 + 192.168.0.0/16 to host")
else:
    print("  ufw inactive -- no firewall change needed")
print()

print("=== 2.6 Resources ===")
print("  cores:", run("nproc"), "(need 16+)")
print("  RAM:", run("free -g | awk '/^Mem:/{print $2}'"), "GB (need 32+)")
print("  disk free:", run("df -h / | tail -1 | awk '{print $4\" / \"$2}'"), "(need 200GB+)")
print("\nPrerequisites check complete.")

## 3. NGC CLI

Installs the NGC CLI if missing and configures it with your API key + org.


In [ ]:
import shutil, os, platform

if shutil.which("ngc"):
    print("NGC CLI:", run("ngc --version 2>&1 | head -1"))
else:
    arch = platform.machine()
    zipname = "ngccli_arm64.zip" if arch in ("aarch64", "arm64") else "ngccli_linux.zip"
    url = f"https://api.ngc.nvidia.com/v2/resources/nvidia/ngc-apps/ngc_cli/versions/4.13.0/files/{zipname}"
    print("Installing NGC CLI 4.13.0 ...")
    # -f fails on HTTP errors, -L follows the NGC CDN redirect, --retry covers
    # transient blips. Verify the payload is really a zip before unzipping: a silent
    # error page saved as ngccli.zip would otherwise fail unzip with a cryptic
    # "End-of-central-directory signature not found".
    ok = False
    for attempt in (1, 2, 3):
        run(f"curl -fSL --retry 3 --retry-delay 3 -o /tmp/ngccli.zip {shlex.quote(url)}", check=False)
        if "Zip archive" in run("file -b /tmp/ngccli.zip 2>/dev/null", check=False):
            ok = True
            break
        print(f"  attempt {attempt}: download was not a valid zip; retrying ...")
        run("sleep 5", check=False)
    assert ok, ("NGC CLI download did not return a valid zip (network/redirect issue reaching "
                "xfiles.ngc.nvidia.com). Re-run this cell, or install ngc manually.")
    run("sudo mkdir -p /usr/local/lib && sudo unzip -qo /tmp/ngccli.zip -d /usr/local/lib")
    run("sudo chmod +x /usr/local/lib/ngc-cli/ngc && sudo ln -sfn /usr/local/lib/ngc-cli/ngc /usr/local/bin/ngc")
    print("Installed:", run("ngc --version 2>&1 | head -1"))

# Configure org only if explicitly provided or no config exists yet
_cfg = os.path.expanduser("~/.ngc/config")
if NGC_CLI_ORG or not os.path.isfile(_cfg):
    org = NGC_CLI_ORG or "nvidia"
    os.makedirs(os.path.dirname(_cfg), exist_ok=True)
    with open(_cfg, "w") as f:
        f.write(f"[CURRENT]\napikey = {NGC_CLI_API_KEY}\nformat_type = ascii\norg = {org}\n")
    print(f"NGC CLI configured (org={org})")
print(run("ngc config current 2>/dev/null | grep -viE 'apikey|\\*\\*\\*'", check=False))

## 4. Docker Login

Authenticate with `nvcr.io` and verify access to the Halos artifacts.


In [ ]:
import subprocess
r = subprocess.run(["docker","login","nvcr.io","-u","$oauthtoken","--password-stdin"],
                   input=NGC_CLI_API_KEY, capture_output=True, text=True)
print("docker login nvcr.io:", "OK" if r.returncode==0 else "FAILED\n"+r.stderr)
if r.returncode != 0:
    raise RuntimeError("docker login failed -- check NGC key access")

print("\n=== Access checks ===")
for img in (ISAAC_SIM_IMAGE, PSF_IMAGE):
    ok = subprocess.run(["bash","-c",f"docker manifest inspect {shlex.quote(img)} >/dev/null 2>&1"]).returncode==0
    print(("PASS" if ok else "FAIL"), "image", img)
ok = subprocess.run(["bash","-c",f"ngc registry resource info {shlex.quote(MDX_DATA_RESOURCE)} >/dev/null 2>&1"]).returncode==0
print(("PASS" if ok else "FAIL"), "resource", MDX_DATA_RESOURCE)

## 5. Get Deployment Artifacts

Compose files for **both** stacks ship in-tree (the cloned repos). From NGC we fetch:
- **VSS app-data** (videos + perception models) → `vss-warehouse-app-data`
- **Halos sil-data** (Isaac scenes, `collected-assets/`, comm-layer/psf logs dirs) → `sil-data`

and pre-pull the **PSF** + **Isaac Sim** images (in parallel — saves wall time).

> First run only — already-present directories are reused. `chmod -R 777` on the app-data dir is **required** (Redis/Kafka/Elasticsearch write into it as different UIDs).


In [ ]:
import os, subprocess, shlex
_base = DOWNLOAD_DIR.rstrip("/") if DOWNLOAD_DIR else os.path.expanduser("~")

# ---- VSS app-data ----
_app_ver = APP_DATA_RESOURCE.split(":", 1)[-1]
_app_extract = os.path.join(_base, f"vss-warehouse-app-data_v{_app_ver}")
APP_DATA_DIR = os.path.join(_app_extract, "vss-warehouse-app-data")
# fall back to a pre-existing extract anywhere obvious
for cand in (APP_DATA_DIR, os.path.expanduser("~/halos/vss-warehouse-app-data")):
    if os.path.isdir(cand):
        APP_DATA_DIR = cand
        break
if os.path.isdir(APP_DATA_DIR):
    print(f"VSS app-data present: {APP_DATA_DIR}")
else:
    print(f"Downloading VSS app-data ({APP_DATA_RESOURCE}) ...")
    run(f"ngc registry resource download-version {shlex.quote(APP_DATA_RESOURCE)}", cwd=_base)
    run(f"tar -xf {shlex.quote(os.path.join(_app_extract, 'vss-warehouse-app-data.tar.gz'))}", cwd=_app_extract)
    print("Extracted.")
run(f"sudo chmod -R 777 {shlex.quote(APP_DATA_DIR)}", check=False)
print(f"  APP_DATA_DIR: {APP_DATA_DIR}  (chmod 777)")

# ---- Halos sil-data ----
for cand in (MDX_DATA_DIR, os.path.expanduser("~/halos/sil-data")):
    if os.path.isdir(os.path.join(cand, "collected-assets")):
        MDX_DATA_DIR = cand
        break
if os.path.isdir(os.path.join(MDX_DATA_DIR, "collected-assets")):
    print(f"\nHalos sil-data present: {MDX_DATA_DIR}")
else:
    print(f"\nDownloading Halos sil-data ({MDX_DATA_RESOURCE}) ...")
    run(f"ngc registry resource download-version {shlex.quote(MDX_DATA_RESOURCE)}", cwd="/tmp")
    # tarball extracts into a sil-data/ folder at the parent of MDX_DATA_DIR
    tb = run("ls /tmp/sample-sil-data_v*/*.tar.gz /tmp/*sil-data*/*.tar.gz 2>/dev/null | head -1", check=False)
    assert tb, "Could not locate downloaded sil-data tarball under /tmp"
    run(f"tar -xzf {shlex.quote(tb)} --directory={shlex.quote(os.path.dirname(MDX_DATA_DIR))}")
    print("Extracted.")
assert os.path.isdir(os.path.join(MDX_DATA_DIR, "collected-assets")), f"sil-data missing collected-assets at {MDX_DATA_DIR}"
print(f"  MDX_DATA_DIR: {MDX_DATA_DIR}")
print("  collected-assets:", run(f"ls {shlex.quote(os.path.join(MDX_DATA_DIR, 'collected-assets'))}"))

# ---- Pre-pull PSF + Isaac in parallel ----
print("\nPre-pulling PSF + Isaac Sim images (parallel) ...")
prepull = (f"docker pull {shlex.quote(ISAAC_SIM_IMAGE)} & P1=$!; "
           f"docker pull {shlex.quote(PSF_IMAGE)} & P2=$!; "
           "wait $P1; echo ISAAC_RC=$?; wait $P2; echo PSF_RC=$?")
print(run(f"bash -c {shlex.quote(prepull)} 2>&1 | grep -E 'ISAAC_RC|PSF_RC|Status:' | tail -4", check=False))
print("\nArtifacts ready.")

## 6. Detect Network Configuration

Auto-detects internal (`HOST_IP`) and external (`EXTERNAL_IP`) addresses, and Brev secure-link info (`BREV_ENV_ID`). On Brev, browser traffic routes through the HAProxy ingress on a single port (7777) via a Cloudflare-fronted secure link.


In [ ]:
import os, subprocess

def _internal_ip():
    return run("ip route get 1.1.1.1 | awk '/src/{for(i=1;i<=NF;i++) if($i==\"src\") print $(i+1)}'", check=False)

def _external_ip():
    for c in ("curl -s --max-time 5 ifconfig.me", "curl -s --max-time 5 icanhazip.com"):
        v = run(c, check=False)
        if v:
            return v
    return ""

def _etc_env():
    d = {}
    if os.path.isfile("/etc/environment"):
        for line in open("/etc/environment"):
            line = line.strip()
            if "=" in line and not line.startswith("#"):
                k, _, v = line.partition("=")
                d[k.strip()] = v.strip().strip('"')
    return d

HOST_IP     = HOST_IP_OVERRIDE or _internal_ip()
EXTERNAL_IP = EXTERNAL_IP_OVERRIDE or _external_ip()
print(f"HOST_IP (internal):  {HOST_IP}")
print(f"EXTERNAL_IP:         {EXTERNAL_IP}")

BREV_ENV_ID = os.environ.get("BREV_ENV_ID") or _etc_env().get("BREV_ENV_ID", "")
PROXY_PORT  = os.environ.get("PROXY_PORT", "7777")
if BREV_ENV_ID:
    os.environ["BREV_ENV_ID"] = BREV_ENV_ID
    print(f"\nBrev detected: BREV_ENV_ID={BREV_ENV_ID}")
    print(f"  Secure link (HAProxy {PROXY_PORT}): https://{PROXY_PORT}-{BREV_ENV_ID}.brevlab.com")
else:
    print("\nNo Brev env detected -- will use direct IP for UI access.")
assert HOST_IP, "Could not detect HOST_IP -- set HOST_IP_OVERRIDE in Section 1."

## 7. Apply VSS SIL Overrides

Applied **before** deploying VSS — if VSS builds its TensorRT engine with the wrong config, a restart costs another ~15 min.

1. **`warehouse-operations/.env`** → `bp_wh_kafka` (PSF needs Kafka), `LLM_MODE/VLM_MODE=none`, 3-cam synthetic dataset, `NUM_STREAMS=3`, extended profile, hardware profile, paths, IPs (+ Brev secure-link overrides if applicable).
2. **`ds-main-config.txt`** → disable SEI extraction + `attach-sys-ts-as-ntp=1` (Isaac 6.0 embeds SEI, but PSF currently doesn't support sim time, so using it as the timestamp makes PSF drop events as STALE).
3. **`vst_config.json`** → `bbox_tolerance_ms=100` (reduces bbox flicker).


In [ ]:
import re, shlex, os

# ---- 7.1 warehouse .env ----
def _merge_env(path, updates):
    lines = open(path, encoding="utf-8").readlines()
    seen, out = set(), []
    for line in lines:
        s = line.strip()
        if s and not s.startswith("#") and "=" in s:
            k = s.split("=", 1)[0].strip()
            if k in updates:
                out.append(f"{k}={shlex.quote(updates[k])}\n"); seen.add(k); continue
        out.append(line)
    for k, v in updates.items():
        if k not in seen:
            out.append(f"{k}={shlex.quote(v)}\n")
    open(path, "w", encoding="utf-8").writelines(out)

_env = {
    "MODE": "2d", "BP_PROFILE": "bp_wh_kafka", "STREAM_TYPE": "kafka",
    "LLM_MODE": "none", "VLM_MODE": "none",
    "SAMPLE_VIDEO_DATASET": "warehouse-loading-dock-3cams-synthetic", "NUM_STREAMS": "3",
    "MINIMAL_PROFILE": "", "HARDWARE_PROFILE": HARDWARE_PROFILE,
    "VSS_APPS_DIR": VSS_DEPLOY_DIR, "VSS_DATA_DIR": APP_DATA_DIR,
    "HOST_IP": HOST_IP, "EXTERNAL_IP": EXTERNAL_IP or HOST_IP,
    "NGC_CLI_API_KEY": NGC_CLI_API_KEY,
}
if BREV_ENV_ID:
    _bh = f"{PROXY_PORT}-{BREV_ENV_ID}.brevlab.com"
    _env.update({"HAPROXY_PORT": PROXY_PORT, "VSS_PUBLIC_HTTP_PROTOCOL": "https",
                 "VSS_PUBLIC_WS_PROTOCOL": "wss", "VSS_PUBLIC_HOST": _bh, "VSS_PUBLIC_PORT": "443"})
_merge_env(WAREHOUSE_ENV, _env)
print(f"[.env] bp_wh_kafka, none/none, 3 cams, HW={HARDWARE_PROFILE}" + (", Brev secure-link" if BREV_ENV_ID else ""))

# ---- 7.2 ds-main-config.txt ----
txt = open(DS_MAIN_CONFIG).read()
for key in ("extract-sei-type5-data", "sei-uuid", "extract-sei-sim-time", "drop-backward-sei"):
    txt = re.sub(rf"(?m)^(?!\s*#)\s*({re.escape(key)}\s*=.*)$", r"# \1", txt)
txt = re.sub(r"(?m)^\s*#?\s*attach-sys-ts-as-ntp\s*=.*$", "attach-sys-ts-as-ntp=1", txt)
open(DS_MAIN_CONFIG, "w").write(txt)
print("[ds-main-config] SEI extraction disabled; attach-sys-ts-as-ntp=1")

# ---- 7.3 vst_config.json ----
v = open(VST_CONFIG).read()
v = re.sub(r'("bbox_tolerance_ms"\s*:\s*)\d+', r"\g<1>100", v)
open(VST_CONFIG, "w").write(v)
print("[vst_config] bbox_tolerance_ms=100")

# ---- 7.5 haproxy ingress: allow Brev secure-link vhosts (works for any VM/ID) ----
# The ingress 404s any Host not in its allow-list. Brev secure links look like
# <name>-<BREV_ENV_ID>.apps.run.brev.nvidia.com (or 7777-<id>.brevlab.com); the exact
# host changes per VM, so allow the whole secure-link domains by suffix (hdr_end).
if BREV_ENV_ID:
    _hap = os.path.join(VSS_DEPLOY_DIR, "services", "infra", "haproxy", "haproxy.cfg.template")
    if not os.path.isfile(_hap):
        print("[haproxy] template not found -- skipped:", _hap)
    elif "apps.run.brev.nvidia.com" in open(_hap).read():
        print("[haproxy] secure-link vhosts already allowed")
    else:
        _out = []
        for _ln in open(_hap).read().splitlines(keepends=True):
            if _ln.strip() == "http-request deny deny_status 404 if !known_host":
                _out.append("    acl known_host hdr_end(host) -i .brevlab.com\n")
                _out.append("    acl known_host hdr_end(host) -i .apps.run.brev.nvidia.com\n")
            _out.append(_ln)
            if 'acl h_main hdr(host) -i "127.0.0.1:${HAPROXY_PORT}"' in _ln:
                _out.append("    acl h_main hdr_end(host) -i .brevlab.com\n")
                _out.append("    acl h_main hdr_end(host) -i .apps.run.brev.nvidia.com\n")
        open(_hap, "w").write("".join(_out))
        print("[haproxy] secure-link vhosts allowed (.brevlab.com + .apps.run.brev.nvidia.com)")

print("\nVSS SIL overrides applied.")

## 8. Deploy VSS Warehouse 3.2 (Stack 1 — perception)

Brings up the perception backend from `<vss_repo>/deploy/docker`. On first run this pulls the VSS image set and the perception container compiles its TensorRT engine. `heal_infra_perms()` auto-repairs the Kafka/Redis/Elasticsearch permission crash-loop if it occurs.

**Expected: ~10–15 min** first run. Full log: `~/deploy_vss.log`.


In [ ]:
import os
_rc = stream_compose(
    ["docker","compose","--progress","plain","-f","compose.yml",
     "--env-file","industry-profiles/warehouse-operations/.env",
     "up","--detach","--pull","always","--force-recreate","--build"],
    cwd=VSS_DEPLOY_DIR, log_file=os.path.expanduser("~/deploy_vss.log"))

# Auto-heal the data-dir permission crash-loop, then bring up any deps that failed
if not heal_infra_perms():
    print("  WARNING: infra still unhealthy -- check docker logs kafka/redis/elasticsearch")
print("\nRe-running compose up to start any services that waited on infra ...")
_rc2 = stream_compose(
    ["docker","compose","--progress","plain","-f","compose.yml",
     "--env-file","industry-profiles/warehouse-operations/.env","up","--detach"],
    cwd=VSS_DEPLOY_DIR, log_file=os.path.expanduser("~/deploy_vss2.log"))
print("\n=== containers ===")
print(run("docker ps --format '{{.Names}}\t{{.Status}}' | sort"))

## 9. Verify VSS Perception Ready

Two ready signals (do **not** deploy Halos until both pass):
1. `vss-rtvi-cv` produces FPS on **all 3** cameras (TensorRT engine built)
2. Kafka `mdx-events` topic has data (perception → behavior-analytics → Kafka)

DeepStream boots with **0 sources** and is provisioned at runtime over its REST API (`:9000`); the SDR/WDM only pushes sources on a *new* VST sensor event. After a **Stop/Teardown redeploy** that event doesn't re-fire, so this cell **self-heals**: if fewer than 3 cameras are live (FPS > 0) it adds the nvstreamer sample streams straight to the REST API — and any source stuck at 0 FPS is removed + re-added.


In [ ]:
import re

def _cam_fps():
    """Most-recent current-FPS per source_id from DeepStream PERF lines."""
    out = run("docker logs --tail 600 vss-rtvi-cv 2>&1 | grep 'stream_name Camera'", check=False)
    fps = {}
    for ln in out.splitlines():
        m = re.search(r"([\d.]+)\s*\(([\d.]+)\)\s+source_id\s*:\s*(\d+)", ln)
        if m:
            fps[m.group(3)] = float(m.group(1))  # keep last (newest) reading per source
    return fps

def _cam_status():
    """name -> latest current-FPS, parsed from DeepStream PERF lines (no regex)."""
    out = run("docker logs --tail 600 vss-rtvi-cv 2>&1 | grep 'stream_name Camera'", check=False)
    st = {}
    for ln in out.splitlines():
        if "stream_name " not in ln:
            continue
        name = ln.split("stream_name ", 1)[1].split()[0]
        try:
            st[name] = float(ln.split("(", 1)[0].split()[-1])  # leading current-FPS token; keep newest
        except Exception:
            pass
    return st

def _live_names():
    return {n for n, f in _cam_status().items() if f > 0}

def _cams_live():
    st = _cam_status(); live = _live_names()
    print(f"    cameras: {len(live)}/{len(st)} live  live={sorted(live)}  fps=" + str({n: st[n] for n in sorted(st)}))
    return len(live) >= 3

# --- perception source readiness + self-heal ---------------------------------
# DeepStream starts with num-source-bins=0 and is provisioned at runtime over its REST API
# (:9000). The SDR/WDM only pushes sources on a NEW sensor event, so a freshly-recreated
# perception (after a Stop/Teardown redeploy) can sit forever at `Active sources : 0`. If the
# cameras don't come live on their own, add the nvstreamer sample streams straight to the REST
# API (one at a time -- batching stalls caps negotiation). Also fixes the partial cold-start race.
import json, urllib.request
NVSTREAMER_HTTP_PORT = "31000"      # VSS default; serves the sample-video RTSP list
DS_REST = "http://localhost:9000"   # vss-rtvi-cv nvmultiurisrcbin REST

def _ds_ready():
    try:
        return "YES" in urllib.request.urlopen(f"{DS_REST}/api/v1/ready", timeout=5).read().decode()
    except Exception:
        return False

def _nvstreamer_streams():
    """[(camera_id, rtsp_url)] of the sample videos served by nvstreamer."""
    sj = json.load(urllib.request.urlopen(
        f"http://localhost:{NVSTREAMER_HTTP_PORT}/api/v1/sensor/streams", timeout=10))
    out = []
    for entry in sj:
        for sid, arr in entry.items():
            main = next((s for s in arr if s.get("isMain")), (arr[0] if arr else None))
            if main and main.get("url"):
                out.append((main.get("name", sid), main["url"]))
    return out

def _ds_post(path, cid, url, change):
    body = json.dumps({"key": "sensor", "value": {
        "camera_id": cid, "camera_name": cid, "camera_url": url,
        "change": change, "metadata": {}}}).encode()
    req = urllib.request.Request(f"{DS_REST}{path}", data=body,
                                 headers={"Content-Type": "application/json"}, method="POST")
    try:
        return str(urllib.request.urlopen(req, timeout=30).getcode())
    except Exception as e:
        return f"ERR {e}"

def _provision_sources():
    """Add sample streams straight to the perception REST. A name present in PERF but stuck at
    0 FPS is NOT live, so remove it first (can't re-add a duplicate camera_id) then add fresh."""
    try:
        streams = _nvstreamer_streams()
    except Exception as e:
        print("    [provision] cannot read nvstreamer streams:", e); return
    st = _cam_status(); live = _live_names()
    print(f"    [provision] nvstreamer={[c for c, _ in streams]} live={sorted(live)} present={sorted(st)}")
    for cid, url in streams:
        if cid in live:
            continue
        if cid in st:  # present but 0 FPS -> reset the stuck source so the re-add takes
            print(f"      reset {cid} (stuck 0fps): remove -> {_ds_post('/api/v1/stream/remove', cid, url, 'camera_remove')}")
            time.sleep(3)
        print(f"      add {cid}: HTTP {_ds_post('/api/v1/stream/add', cid, url, 'camera_add')}")
        time.sleep(20)

# Wait for the perception REST (covers the cold TensorRT engine build), give the SDR/WDM a
# short window to auto-provision, then self-heal by adding the sources directly if needed.
poll(lambda: _ds_ready() or _cams_live(), timeout_s=1200, interval_s=20, desc="perception REST ready (:9000)")
ok1 = poll(_cams_live, timeout_s=240, interval_s=15, desc="3 cameras producing FPS")
if not ok1:
    print("  <3 cameras live -- SDR/WDM did not (re)provision this DeepStream "
          "(normal after a Stop/Teardown redeploy); adding sources via REST :9000 ...")
    _provision_sources()
    ok1 = poll(_cams_live, timeout_s=600, interval_s=20, desc="3 cameras producing FPS (after provision)")
if ok1:
    print(run("docker logs vss-rtvi-cv 2>&1 | grep -E 'PERF|stream_name Camera' | tail -4", check=False))

def _kafka_events():
    return subprocess.run(["bash","-c",
        "docker exec kafka kafka-console-consumer --bootstrap-server localhost:9092 "
        "--topic mdx-events --max-messages 1 --timeout-ms 30000 >/dev/null 2>&1"]).returncode == 0

ok2 = poll(_kafka_events, timeout_s=600, interval_s=15, desc="Kafka mdx-events has data")
assert ok1 and ok2, ("VSS perception not ready: need all 3 cameras at FPS>0 AND Kafka mdx-events. "
                     "Check `docker logs vss-rtvi-cv` (Active sources) and that nvstreamer lists 3 "
                     f"streams at http://localhost:{NVSTREAMER_HTTP_PORT}/api/v1/sensor/streams.")
print("\nVSS perception READY -- proceed to Halos SIL.")

## 10. Configure Halos `sil.env`

Fills the `# change me` placeholders in `deployments/profiles/sil.env`:
`HOST_IP`, `MDX_SAMPLE_APPS_DIR`, `MDX_DATA_DIR`, `DOCKER_GID` (host docker group), `ISAAC_GPU_DEVICE`, `ROS_DOMAIN_ID`. `PSF_IMAGE`/`ISAAC_SIM_IMAGE` stay as the template defaults.


In [ ]:
_docker_gid = run("getent group docker | cut -d: -f3", check=False) or "999"
_sil = {
    "HOST_IP": f"'{HOST_IP}'",
    "MDX_SAMPLE_APPS_DIR": HALOS_REPO_PATH,
    "MDX_DATA_DIR": MDX_DATA_DIR,
    "DOCKER_GID": _docker_gid,
    "ISAAC_GPU_DEVICE": str(ISAAC_GPU_DEVICE),
    "ROS_DOMAIN_ID": str(ROS_DOMAIN_ID),
}
lines = open(SIL_ENV).readlines()
seen, out = set(), []
for line in lines:
    s = line.strip()
    if s and not s.startswith("#") and "=" in s:
        k = s.split("=", 1)[0].strip()
        if k in _sil:
            out.append(f"{k}={_sil[k]}\n"); seen.add(k); continue
    out.append(line)
for k, v in _sil.items():
    if k not in seen:
        out.append(f"{k}={v}\n")
open(SIL_ENV, "w").writelines(out)
print("sil.env configured:")
for k, v in _sil.items():
    print(f"  {k}={v}")

## 11. Deploy Halos SIL (Stack 2)

`setup.sh sil` creates data dirs/logs, `cleanup_all_datalog.sh sil` clears prior run logs, then `docker compose up -d --build` (comm-layer + isaac-sim build from local Dockerfiles).

**Expected: ~3–5 min** (image builds; Isaac scene load happens when the scenario runs).


In [ ]:
import os
print(run(f"{shlex.quote(os.path.join(HALOS_REPO_PATH, 'closed-loop-testing/scripts/setup.sh'))} sil", cwd=HALOS_DEPLOY_DIR, check=False))
print(run(f"{shlex.quote(os.path.join(HALOS_REPO_PATH, 'closed-loop-testing/scripts/cleanup_all_datalog.sh'))} sil", cwd=HALOS_DEPLOY_DIR, check=False))
_rc = stream_compose(
    ["docker","compose","--progress","plain","--env-file","profiles/sil.env","up","-d","--build"],
    cwd=HALOS_DEPLOY_DIR, log_file=os.path.expanduser("~/deploy_halos.log"))
import time; time.sleep(8)
print("\n=== Halos services ===")
print(run("docker ps --format '{{.Names}}\t{{.Status}}' | grep -E 'safety-core|comm-layer|isaac-sim|forklift-controller'", check=False))
# cleanup_all_datalog.sh does `rm -rf comm-layer/*`, deleting opc_server.log out from under an
# already-running comm-layer (re-run case); `up -d` won't recreate a running container, so the
# UDP->OPC log is never re-touched and Section 12 false-fails. If it's gone, restart comm-layer
# to recreate its logs (pss.log is truncated, not deleted, so PSF is unaffected).
if not os.path.isfile(os.path.join(MDX_DATA_DIR, "comm-layer/opc_server.log")):
    print("  opc_server.log missing (cleanup wiped a running comm-layer) -- restarting comm-layer")
    run("docker restart comm-layer", check=False); time.sleep(6)
    print(run("docker ps --format '{{.Names}}\t{{.Status}}' | grep comm-layer", check=False))

## 12. Verify Halos Services + PSF Wiring

- All 4 services `Up`
- PSF → comm-layer wired (`opc_server.log` is non-empty)
- ROS isolation: `Publisher count: 1` for `/safety/is_muted` (the CLI may need the Isaac scene up to report subscriber count; the data path is confirmed by `ros_bridge.log`)


In [ ]:
import os, time
ok = poll(lambda: os.path.isfile(os.path.join(MDX_DATA_DIR, "comm-layer/opc_server.log"))
                  and os.path.getsize(os.path.join(MDX_DATA_DIR, "comm-layer/opc_server.log")) > 0,
          timeout_s=180, interval_s=5, desc="PSF -> comm-layer wired (opc_server.log)")
print("\n=== opc_server.log (tail) ===")
print(run(f"tail -n 6 {shlex.quote(os.path.join(MDX_DATA_DIR, 'comm-layer/opc_server.log'))}", check=False))
print("\n=== ROS isolation ===")
print(run("timeout 25 docker exec comm-layer bash -lc 'source /opt/ros/jazzy/setup.bash && ros2 topic info /safety/is_muted -v 2>/dev/null | grep -E \"Publisher count|Subscription count\"' || echo '(topic appears once Isaac subscribes -- Section 13)'", check=False))

## 13. Run the Isaac Sim Scenario

Launches the forklift scenario (detached), then drives the **Isaac→VSS handoff** so perception switches off the sample loop and onto the live sim feed:

1. **Wait for Isaac to serve** all 3 cameras over RTSP — Isaac Sim 6.0 self-hosts one RTSP server per camera (`:8554`/`:8555`/`:8556`, from `cameras.yaml`). First run includes scene load + RT shader compile (**~5–12 min**, cached afterwards).
2. **Swap sources sample→Isaac.** Isaac registers its cameras to VST under the same names as the sample feed (`Camera`/`Camera_01`/`Camera_02`); on a clean deploy the SDR/WDM may swap the DeepStream sources automatically. If it doesn't land (e.g. a **Stop/Teardown redeploy** where the WDM is disconnected — same reason Section 9 needs a manual provision) the cell performs the swap explicitly over the perception REST (`:9000`): for each live source it removes it by its real `(camera_id, camera_url)` — recovering the URL WDM used (`rtsp://…/live/<camera_id>`) from the sdr-controller log, or the nvstreamer sample URL for a self-healed source — then adds the Isaac RTSP url under the same camera name.
3. **Verify** the handoff by `ffprobe`-ing each per-camera RTSP endpoint (the same probe as `scripts/rtsp/check_postplay.sh`) and confirming DeepStream reports per-camera FPS > 0.

Set `SIMULATION_DURATION` in Section 1 (>0, seconds) for a longer watchable run.


In [ ]:
import os, time
# ffprobe (from ffmpeg) checks the Isaac RTSP endpoints in this cell's readiness loop; Section 15.1
# also installs ffmpeg but runs later, so ensure it's present here (fresh VM would have none).
run("which ffprobe >/dev/null 2>&1 || sudo apt-get install -y ffmpeg", check=False)
# Optional: lengthen the run for a longer watchable scenario
if SIMULATION_DURATION and SIMULATION_DURATION > 0:
    run(f"docker exec -u 0 isaac-sim bash -lc \"sed -i -E 's/^([[:space:]]*simulation_duration:).*/\\1 {SIMULATION_DURATION}/' /isaac-sim/sil/configs/default_config_ros.yaml\" || true", check=False)

print("Launching Isaac scenario (detached) ...")
run("docker exec -d isaac-sim bash -lc 'cd /isaac-sim/sil/scripts && "
    "./run_sdg.sh -c /isaac-sim/sil/configs/default_config_ros.yaml --start --headless "
    "--enable-vst --cameras-config /isaac-sim/sil/configs/cameras.yaml'", check=False)
time.sleep(10)
print("run process:", run("docker exec isaac-sim pgrep -af run_actor_sdg.py | head -1", check=False) or "(not found)")
_KIT = "ls -t /isaac-sim/.nvidia-omniverse/logs/Kit/*/*/kit_*.log 2>/dev/null | head -1"
print("Isaac log:  docker exec isaac-sim bash -lc 'tail -f \"$(" + _KIT + ")\"'")
print(run("docker exec isaac-sim bash -lc 'tail -n 15 \"$(" + _KIT + ")\"'", check=False) or "  (Isaac kit log not found yet)")

# --- Isaac -> VSS handoff -----------------------------------------------------
# Isaac Sim 6.0 self-hosts one RTSP server per camera (isaacsim.streaming.rtsp); the per-camera
# ports/mounts come from cameras.yaml (Camera->8554/camera, Camera_01->8555/camera_01, ...). On a
# clean deploy the SDR/WDM may swap the DeepStream sources (sample -> Isaac) on its own. If it
# doesn't land (e.g. a Stop/Teardown redeploy where the WDM is disconnected -- same reason Section 9
# needs a manual provision) we swap explicitly over the REST API (:9000): remove each live source by
# its real (camera_id, url) -- the url is NOT in get-stream-info, so recover it from the
# sdr-controller log (rtsp://.../live/<id>) or the nvstreamer sample list -- then add the Isaac url
# under the same camera name. Readiness is ffprobe reaching each per-camera RTSP endpoint (same
# probe as scripts/rtsp/check_postplay.sh) plus DeepStream reporting per-camera FPS > 0.
import json, urllib.request, re, shlex, subprocess
DS_REST = "http://localhost:9000"
# Per-camera Isaac RTSP endpoints parsed from cameras.yaml: {name: rtsp://HOST_IP:port/mount}.
_cam_yaml = os.path.join(HALOS_REPO_PATH, "closed-loop-testing", "isaac-sim", "sil", "configs", "cameras.yaml")
_cam_txt = re.sub(r"(?m)#.*$", "", open(_cam_yaml).read())  # strip comments: a commented rtsp.port adds a 4th match and misaligns the zip
_names = re.findall(r"-\s*name:\s*(\S+)", _cam_txt)
_ports = re.findall(r"\bport:\s*(\d+)", _cam_txt)
_mounts = re.findall(r"\bmount_path:\s*(\S+)", _cam_txt)
assert _names and len(_names) == len(_ports) == len(_mounts), f"cameras.yaml parse misaligned: names={_names} ports={_ports} mounts={_mounts}"
ISAAC_RTSP = {n: f"rtsp://{HOST_IP}:{p}{m}" for n, p, m in zip(_names, _ports, _mounts)}
print("    Isaac RTSP endpoints:", ISAAC_RTSP)

def _ffprobe_ok(url):
    # RTSP endpoint is live if ffprobe reads a stream from it (mirrors check_postplay.sh C2).
    return subprocess.run(["bash", "-c", f"timeout 10 ffprobe -v error -show_streams -of json {shlex.quote(url)}"],
                          capture_output=True, text=True).returncode == 0

def _isaac_pub():
    # Isaac self-hosts RTSP (no mediamtx): a camera is "publishing" when its per-camera endpoint
    # answers ffprobe. Returns the set of camera names currently serving.
    return {name for name, url in ISAAC_RTSP.items() if _ffprobe_ok(url)}

def _ds_sources():
    """[(camera_id, camera_name)] currently on the perception pipeline (None if endpoint absent).
    This build nests the list under stream-info.stream-info (older docs say stream-list) and does
    NOT return camera_url -- so the swap recovers each url separately via _registered_url()."""
    try:
        info = json.load(urllib.request.urlopen(
            f"{DS_REST}/api/v1/stream/get-stream-info", timeout=8)).get("stream-info", {})
        lst = info.get("stream-info") or info.get("stream-list") or []
        return [(s.get("camera_id"), s.get("camera_name") or s.get("camera_id")) for s in lst]
    except Exception:
        return None

def _registered_url(cid, name):
    """The URL a source was registered with, so /stream/remove can match (it needs the exact url).
    WDM registers a VST proxy url (rtsp://host:port/live/<camera_id>) -- recover it from the
    sdr-controller log; a Section-9 self-healed source instead uses the nvstreamer url by name."""
    u = run(f"docker logs sdr-controller 2>&1 | grep -oE 'rtsp://[^/ ]+/live/{cid}' | tail -1", check=False).strip()
    if u:
        return u
    try:
        nv = dict(_nvstreamer_streams())
    except Exception:
        nv = {}
    return nv.get(name) or nv.get(cid)

def _isaac_ready():
    # No mediamtx to inspect: the live-feed-is-Isaac signal is all 3 per-camera RTSP endpoints
    # answering ffprobe AND DeepStream reporting FPS>0 on 3 sources.
    serving = _isaac_pub(); live = sum(1 for f in _cam_status().values() if f > 0)
    print(f"    Isaac RTSP endpoints up={len(serving)}/3; cameras live={live}; GPU{ISAAC_GPU_DEVICE} "
          + run(f"nvidia-smi --query-gpu=utilization.gpu --format=csv,noheader -i {ISAAC_GPU_DEVICE}", check=False))
    return len(serving) >= 3 and live >= 3

def _swap_to_isaac():
    """Replace whatever is live with the 3 Isaac feeds over the perception REST. Remove each
    current source by its real (camera_id, url) so the remove matches, then add the Isaac url
    under the same camera name (camera_id == name, matching the sample identity)."""
    src = _ds_sources() or []
    if not src:                                   # perception empty (bad redeploy) -> just add
        src = [(n, n) for n in ("Camera", "Camera_01", "Camera_02")]
    for cid, name in src:
        url = _registered_url(cid, name)
        if url and url not in ISAAC_RTSP.values():  # a sample source -> remove it by its real (id,url)
            print(f"      remove {name} [{cid}]: {_ds_post('/api/v1/stream/remove', cid, url, 'camera_remove')}")
            time.sleep(3)
        print(f"      add isaac {name}: {_ds_post('/api/v1/stream/add', name, ISAAC_RTSP.get(name, ''), 'camera_add')}")
        time.sleep(20)

# 1) wait for Isaac to serve all 3 RTSP cameras (first run: scene load + RT shader compile)
poll(lambda: len(_isaac_pub()) >= 3, timeout_s=1800, interval_s=20,
     desc="Isaac serving 3 RTSP cameras")
print("    Isaac RTSP cameras up:", sorted(_isaac_pub()))

# 2) let the SDR/WDM swap on its own, then swap explicitly if it didn't (redeploy with dead WDM)
ok = poll(_isaac_ready, timeout_s=240, interval_s=15, desc="Isaac->VSS handoff (DeepStream ingesting Isaac)")
if not ok:
    print("  Handoff didn't land on its own -- swapping sample->Isaac via REST :9000 ...")
    _swap_to_isaac()
    ok = poll(_isaac_ready, timeout_s=300, interval_s=15, desc="Isaac->VSS handoff (after manual swap)")

assert ok, ("Isaac->VSS handoff failed: DeepStream is not ingesting the 3 Isaac streams. "
            f"Confirm Isaac is serving RTSP (ffprobe {' '.join(ISAAC_RTSP.values())}) and that "
            "those per-camera endpoints are reachable from the host. See the Isaac kit log: "
            "docker exec isaac-sim bash -lc 'tail -120 \"$(ls -t /isaac-sim/.nvidia-omniverse/logs/Kit/*/*/kit_*.log|head -1)\"'")
print("\n=== Isaac RTSP endpoints (ffprobe) ===")
for _n, _u in ISAAC_RTSP.items():
    print(f"  {_n}: {_u}  ->  {'up' if _ffprobe_ok(_u) else 'DOWN'}")
print("=== DeepStream PERF ===")
print(run("docker logs vss-rtvi-cv 2>&1 | grep -E 'Active sources|stream_name Camera' | tail -4", check=False))
print("\nIsaac SIL closed-loop RUNNING -- perception is now on the Isaac feed.")

## 14. Monitor Sim-Driven Safety (MUTE/UNMUTE)

"Working" = **sim-driven** transitions from the forklift cycle (forklift in trailer → MUTE/loading-allowed; human present or exiting → UNMUTE). Re-run this cell to watch live.


In [ ]:
import os, time
_opc = os.path.join(MDX_DATA_DIR, "comm-layer/opc_server.log")
_pss = os.path.join(MDX_DATA_DIR, "psf-log/pss.log")
print("Sampling ~25s of live safety decisions ...")
time.sleep(25)
print("\n=== PSF forklift tripwire events (Isaac-driven) ===")
print(run(f"grep -E 'Forklift tripwire (IN|OUT)' {shlex.quote(_pss)} 2>/dev/null | tail -6", check=False) or "(none yet)")
print("\n=== OPC MUTE/UNMUTE (recent) ===")
print(run(f"grep -E 'MUTE \\(ALLOW|UNMUTE \\(PREVENT' {shlex.quote(_opc)} 2>/dev/null | tail -10", check=False))
print("\n=== counts ===")
print("  MUTE (loading allowed):", run(f"grep -c 'MUTE (ALLOW' {shlex.quote(_opc)} 2>/dev/null", check=False))
print("  UNMUTE (prevent):      ", run(f"grep -c 'UNMUTE (PREVENT' {shlex.quote(_opc)} 2>/dev/null", check=False))
print("\nClosed loop OK when you see BOTH MUTE and UNMUTE transitions tied to the forklift cycle.")

## 15. Access the VST UI

The VST UI shows the Isaac camera feeds with detection overlays; the forklift's safety disc colour follows MUTE/UNMUTE.

**On Brev:** create a secure link for port **7777** (any name) and open it with the **`/vst/`** path. Section 7.5 allow-listed the `*.apps.run.brev.nvidia.com` and `*.brevlab.com` secure-link domains in the ingress, so whatever host Brev assigns works (no per-VM host editing needed).

> **Streaming limitation (live *and* recorded):** VST video playback uses **WebRTC (peer-to-peer UDP)**, which a Brev secure link (TCP/HTTPS only) cannot carry — the UI loads and stream lists/recordings are browsable, but **video frames will not render**. This is a Brev limitation, not a deploy fault. To see frames, **download a clip** (plain HTTP — works through the link) or pull the recorded `.mp4` from disk. The authoritative closed-loop proof is **Section 14** (MUTE/UNMUTE).


In [ ]:
import os
if BREV_ENV_ID:
    base = f"https://{PROXY_PORT}-{BREV_ENV_ID}.brevlab.com"
    print("Access via Brev secure link (create a secure link for port", PROXY_PORT, "):")
    print(f"  VST UI: {base}/vst/")
    print(f"  (Any secure link you create for port {PROXY_PORT} works -- ingress allow-lists the secure-link domains.)")
    print("  Note: live AND recorded video will NOT render through a Brev secure link (WebRTC/UDP);")
    print("        stream lists are browsable -- download a clip to see frames; verify the loop via Section 14.")
else:
    host = EXTERNAL_IP or HOST_IP
    print("Access (direct -- ensure the firewall/security-group exposes the port):")
    print(f"  VST UI (HAProxy): http://{host}:{PROXY_PORT}/vst/")
    print(f"  VST UI (direct):  http://{host}:30888/vst/")
    print(f"\n  SSH tunnel alternative: ssh -L {PROXY_PORT}:localhost:{PROXY_PORT}  then open http://localhost:{PROXY_PORT}/vst/")
print("\nLocal reachability check:")
print("  HAProxy /vst/:", run(f"curl -s -o /dev/null -w '%{{http_code}}' --max-time 8 http://localhost:{PROXY_PORT}/vst/", check=False))
print("  direct  /vst/:", run("curl -s -o /dev/null -w '%{http_code}' --max-time 8 http://localhost:30888/vst/", check=False))

## 15.1 Save camera clips (optional — *see* what Isaac renders)

Brev secure links can't carry live WebRTC video, so to actually watch the sim, grab a few seconds of each Isaac camera straight off its RTSP endpoint (Isaac Sim 6.0 self-hosts RTSP over TCP) into MP4 files and download them from the JupyterLab file browser. **Run this while the Section 13 scenario is live** (cameras publishing). Each clip is also transcoded to H.264 because HEVC often won't play in Chrome.

In [ ]:
import os, re, subprocess, time
CLIP_DIR = os.path.expanduser("~/clips"); os.makedirs(CLIP_DIR, exist_ok=True)
CLIP_SECONDS = 10
run("which ffmpeg >/dev/null 2>&1 || sudo apt-get install -y ffmpeg", check=False)

# Per-camera RTSP endpoints from cameras.yaml (Isaac Sim 6.0 self-hosts one server per camera).
_cam = os.path.join(HALOS_REPO_PATH, "closed-loop-testing", "isaac-sim", "sil", "configs", "cameras.yaml")
_txt = re.sub(r"(?m)#.*$", "", open(_cam).read()) if os.path.isfile(_cam) else ""  # strip comments (a commented rtsp.port would misalign the zip)
_names = re.findall(r"-\s*name:\s*(\S+)", _txt)
_ports = re.findall(r"\bport:\s*(\d+)", _txt)
_mounts = re.findall(r"\bmount_path:\s*(\S+)", _txt)
assert not _names or len(_names) == len(_ports) == len(_mounts), f"cameras.yaml parse misaligned: names={_names} ports={_ports} mounts={_mounts}"
cams = list(zip(_names, _ports, _mounts))

if not cams:
    print("No cameras parsed from cameras.yaml -- check the file / (re)launch Section 13.")
else:
    print("Isaac RTSP cameras:", [f"{n} :{p}{m}" for n, p, m in cams])
    saved = []
    for name, port, mount in cams:
        url = f"rtsp://{HOST_IP}:{port}{mount}"
        live = False
        for _ in range(40):  # wait until the endpoint actually serves
            if subprocess.run(["bash", "-c",
                    f"ffmpeg -rtsp_transport tcp -i {url!r} -t 1 -f null - >/dev/null 2>&1"]).returncode == 0:
                live = True; break
            time.sleep(3)
        if not live:
            print(f"  [{name}] {url} not live -- skipped"); continue
        raw  = os.path.join(CLIP_DIR, f"{name}.mp4")
        h264 = os.path.join(CLIP_DIR, f"{name}_h264.mp4")
        run(f"ffmpeg -y -rtsp_transport tcp -i {url!r} -t {CLIP_SECONDS} -an -c copy {raw!r}", check=False)
        run(f"ffmpeg -y -i {raw!r} -c:v libx264 -pix_fmt yuv420p -an {h264!r}", check=False)  # browser-friendly
        saved.append(h264)
    print("\nSaved (also embedded below; or download from the JupyterLab file browser -> clips/):")
    for s in saved:
        print("  ", s)
    # Play inline so you don't have to open the folder. embed=True base64-inlines the
    # clip, so it renders even when JupyterLab is served through a Brev secure link.
    try:
        from IPython.display import display, Video, HTML
        for s in saved:
            display(HTML(f"<b>{os.path.basename(s)}</b>"))
            display(Video(s, embed=True, width=640))
    except Exception as e:
        print("  (inline preview unavailable:", e, "-- download the file instead)")

## 16. Stop (keep data)

Stops both stacks without deleting volumes/data. Re-run Sections 8 & 11 to restart.

> On a restart the SDR/WDM does **not** re-provision the freshly-recreated DeepStream (it only reacts to *new* VST sensor events). That's expected — **Section 9** self-heals the sample sources and **Section 13** self-heals the Isaac swap, both directly over the perception REST. Just run the notebook top-to-bottom as usual.


In [ ]:
# Manual-only. "Run All" SKIPS this so it never stops a freshly deployed stack.
# To stop both stacks (data kept), set STOP_NOW = True and re-run THIS cell.
STOP_NOW = False
if not STOP_NOW:
    print("[skipped] Section 16 (Stop) is manual-only. Set STOP_NOW = True and re-run this cell to stop both stacks (data is kept).")
else:
    print("Stopping Halos SIL ...")
    print(run("docker compose --env-file profiles/sil.env stop", cwd=HALOS_DEPLOY_DIR, check=False))
    print("Stopping VSS ...")
    print(run("docker compose -f compose.yml --env-file industry-profiles/warehouse-operations/.env stop", cwd=VSS_DEPLOY_DIR, check=False))
    print("Both stacks stopped.")

## 17. Teardown (remove containers + volumes)

Removes containers, networks, and volumes for both stacks. Downloaded data (`sil-data`, app-data) and repos are preserved on disk.


In [ ]:
# Manual-only. "Run All" SKIPS this so it never wipes a freshly deployed stack.
# To remove containers + volumes, set TEARDOWN_NOW = True and re-run THIS cell.
# Downloaded data (sil-data, app-data) and repos are preserved on disk.
TEARDOWN_NOW = False
if not TEARDOWN_NOW:
    print("[skipped] Section 17 (Teardown) is manual-only. Set TEARDOWN_NOW = True and re-run this cell to remove containers + volumes (downloaded data + repos are kept).")
else:
    print("Tearing down Halos SIL ...")
    print(run("docker compose --env-file profiles/sil.env down -v", cwd=HALOS_DEPLOY_DIR, check=False))
    print("Tearing down VSS ...")
    print(run("docker compose -f compose.yml --env-file industry-profiles/warehouse-operations/.env down -v", cwd=VSS_DEPLOY_DIR, check=False))
    print(run("docker volume prune -f", check=False))
    print("Teardown complete (downloaded data + repos preserved).")